# RxGuide AI — Data Cleaning and Exploratory Analysis

This notebook validates, cleans, and explores the pharmaceutical commercial datasets used for HCP targeting and sales-force effectiveness analysis.

## Objectives

- Load and inspect all six datasets
- Validate columns, datatypes, and identifiers
- Detect missing values and duplicate records
- Verify relationships between datasets
- Perform focused exploratory analysis
- Export cleaned datasets for SQL and machine learning

In [2]:
import sys
print(sys.executable)

C:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [3]:
import sys
!{sys.executable} -m pip install matplotlib

  Using cached matplotlib-3.11.1-cp314-cp314-win_amd64.whl.metadata (80 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp314-cp314-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.11.1-cp314-cp314-win_amd64.whl (9.5 MB)
Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl (232 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp314-cp314-win_amd64.whl (2.3 MB)
Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl (75 kB)
Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl (7.2 MB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)

   ---------------------------------------- 0/7 [pyparsing]
   -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Libraries imported successfully.")

Libraries imported successfully.


In [5]:
    # Detect the project root regardless of whether the notebook
# runs from the repository root or the notebooks folder.

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

Project root: d:\rxguide\rxguide-hcp-analytics
Raw data directory: d:\rxguide\rxguide-hcp-analytics\data\raw
Processed data directory: d:\rxguide\rxguide-hcp-analytics\data\processed


In [6]:
file_paths = {
    "hcps": RAW_DATA_DIR / "hcps.csv",
    "sales_reps": RAW_DATA_DIR / "sales_reps.csv",
    "products": RAW_DATA_DIR / "products.csv",
    "call_activity": RAW_DATA_DIR / "call_activity.csv",
    "prescriptions": RAW_DATA_DIR / "prescriptions.csv",
    "ic_quotas": RAW_DATA_DIR / "ic_quotas.csv",
}

missing_files = [
    str(path)
    for path in file_paths.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following dataset files were not found:\n"
        + "\n".join(missing_files)
    )

datasets = {
    name: pd.read_csv(path)
    for name, path in file_paths.items()
}

print("All datasets loaded successfully.\n")

for name, dataframe in datasets.items():
    print(f"{name:<15} -> {dataframe.shape[0]:>6,} rows × "
          f"{dataframe.shape[1]:>2} columns")

All datasets loaded successfully.

hcps            ->    500 rows ×  7 columns
sales_reps      ->     50 rows ×  7 columns
products        ->      5 rows ×  5 columns
call_activity   -> 13,705 rows ×  8 columns
prescriptions   -> 17,329 rows ×  6 columns
ic_quotas       ->    200 rows ×  6 columns


In [7]:
dataset_overview = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": dataframe.shape[0],
            "columns": dataframe.shape[1],
            "missing_values": int(dataframe.isna().sum().sum()),
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "memory_mb": dataframe.memory_usage(deep=True).sum() / (1024 ** 2),
        }
        for name, dataframe in datasets.items()
    ]
)

dataset_overview["memory_mb"] = dataset_overview["memory_mb"].round(3)

dataset_overview

,dataset,rows,columns,missing_values,duplicate_rows,memory_mb
0,hcps,500,7,0,0,0.19
1,sales_reps,50,7,0,0,0.02
2,products,5,5,0,0,0.00
3,call_activity,13705,8,0,0,4.50
4,prescriptions,17329,6,0,0,3.97
5,ic_quotas,200,6,0,0,0.03


In [8]:
for name, dataframe in datasets.items():
    print("=" * 90)
    print(name.upper())
    print("=" * 90)

    print("\nColumns:")
    print(dataframe.columns.tolist())

    print("\nSample records:")
    display(dataframe.head(3))

    print()

HCPS

Columns:
['hcp_id', 'hcp_name', 'specialty', 'territory', 'region', 'segment', 'city']

Sample records:


,hcp_id,hcp_name,specialty,territory,region,segment,city
0,H0001,Dr. HCP_0001,Endocrinologist,North-1,North,B,Delhi
1,H0002,Dr. HCP_0002,Pulmonologist,North-1,North,C,Delhi
2,H0003,Dr. HCP_0003,Endocrinologist,Central-1,Central,C,Chennai



SALES_REPS

Columns:
['rep_id', 'rep_name', 'territory', 'region', 'manager', 'tenure_months', 'hire_date']

Sample records:


,rep_id,rep_name,territory,region,manager,tenure_months,hire_date
0,R001,Rep_001,North-2,North,Rajesh Kumar,38,2021-01-03
1,R002,Rep_002,South-2,South,Priya Sharma,16,2017-08-22
2,R003,Rep_003,North-2,North,Vikram Singh,57,2023-05-26



PRODUCTS

Columns:
['product_id', 'product_name', 'therapy_area', 'launch_year', 'unit_price']

Sample records:


...

,call_id,rep_id,hcp_id,product_id,call_date,call_type,duration_min,samples_given
0,C000001,R001,H0185,P002,2026-04-08,F2F,21,5
1,C000002,R001,H0410,P003,2026-01-25,F2F,17,7
2,C000003,R001,H0300,P003,2025-11-15,F2F,13,0



PRESCRIPTIONS

Columns:
['rx_id', 'hcp_id', 'product_id', 'rx_month', 'rx_count', 'units_dispensed']

Sample records:


,rx_id,hcp_id,product_id,rx_month,rx_count,units_dispensed
0,RX0000001,H0001,P001,2025-05-01,2,98
1,RX0000002,H0001,P001,2025-05-01,2,104
2,RX0000003,H0001,P001,2025-06-01,1,49



IC_QUOTAS

Columns:
['rep_id', 'quarter', 'quota_units', 'actual_units', 'attainment_pct', 'payout_inr']

Sample records:


,rep_id,quarter,quota_units,actual_units,attainment_pct,payout_inr
0,R001,2025-Q2,1719,1555,90.50,67844
1,R001,2025-Q3,1634,1857,113.60,85235
2,R001,2025-Q4,1134,1334,117.60,88227


## Detailed Data Profiling

In this section, each dataset is examined for datatypes, missing values, unique values, numerical distributions, and categorical consistency before any cleaning decisions are made.

In [ ]:
# Create readable variable names for individual datasets

# Create readable variable names for individual datasets
hcps = datasets["hcps"].copy()
sales_reps = datasets["sales_reps"].copy()
products = datasets["products"].copy()
calls = datasets["call_activity"].copy()
prescriptions = datasets["prescriptions"].copy()
quotas = datasets["ic_quotas"].copy()

print("Individual DataFrames created successfully.")
sales_reps = datasets["sales_reps"].copy()
products = datasets["products"].copy()
calls = datasets["call_activity"].copy()
prescriptions = datasets["prescriptions"].copy()
quotas = datasets["ic_quotas"].copy()

print("Individual DataFrames created successfully.")

NameError: name 'datasets' is not defined